In [20]:
import json
from langchain.agents import create_agent
from langchain_community.tools import GoogleSerperResults
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
search_tool = GoogleSerperResults()

def _banner(title: str) -> None:
    line = "=" * 56
    print(f"\n{line}\n {title}\n{line}")

def _truncate(s: str, max_chars: int) -> str:
    return s if len(s) <= max_chars else s[: max_chars - 3] + "..."

def _reasoning(msg: AIMessage) -> str | None:
    raw = (msg.additional_kwargs or {}).get("reasoning_content")
    if isinstance(raw, str) and raw.strip():
        return raw.strip()
    return None

def print_react_update(chunk: dict, observation_max_chars: int = 1800) -> None:
    for node_name, payload in chunk.items():
        if node_name == "model":
            for msg in payload.get("messages", []):
                if not isinstance(msg, AIMessage):
                    continue
                if (r := _reasoning(msg)):
                    _banner("Thinking")
                    print(r)
                if msg.tool_calls:
                    _banner("Action")
                    for tc in msg.tool_calls:
                        print(f"  tool: {tc.get('name', '')}")
                        print(f"  input: {json.dumps(tc.get('args', {}), ensure_ascii=False)}")
                text = (msg.content or "").strip()
                if text and not msg.tool_calls:
                    _banner("Final answer")
                    print(text)
                elif text and msg.tool_calls:
                    _banner("Assistant (partial text)")
                    print(text)
        elif node_name == "tools":
            for msg in payload.get("messages", []):
                if isinstance(msg, ToolMessage):
                    _banner("Observation (tool output)")
                    print(_truncate(str(msg.content), observation_max_chars))

SYSTEM = """You answer using web search when the user asks for current facts, news, weather, prices, or anything time-sensitive.
- Call the tool `google_serper_results_json` with one concise search query (or a second query if the first results are weak).
- Ground every factual claim in the JSON you received: paraphrase snippets and include source titles and URLs when present.
- If the tool returns no useful evidence, say you could not verify and avoid inventing details."""

agent = create_agent(llm, tools=[search_tool], system_prompt=SYSTEM)



In [21]:


question = "What is the weather in Hyderabad today? One short line with a source."

state = {"messages": [HumanMessage(content=question)]}

print("User:", question)
for step in agent.stream(state, stream_mode="updates"):
    print_react_update(step)


User: What is the weather in Hyderabad today? One short line with a source.

 Thinking
We need current weather. Must use web search. Use tool.

 Action
  tool: google_serper_results_json
  input: {"query": "Hyderabad weather today"}

 Observation (tool output)
{'searchParameters': {'q': 'Hyderabad weather today', 'gl': 'us', 'hl': 'en', 'type': 'search', 'num': 10, 'engine': 'google'}, 'answerBox': {'title': '', 'answer': '84°F', 'source': 'Google Weather', 'sourceLink': 'https://support.google.com/websearch/answer/13687874'}, 'organic': [{'title': 'Hyderabad, Telangana, India Weather Forecast - AccuWeather', 'link': 'https://www.accuweather.com/en/in/hyderabad/202190/weather-forecast/202190', 'snippet': 'Hourly Weather · 1 AM 85°. rain drop 0% · 2 AM 83°. rain drop 0% · 3 AM 82°. rain drop 0% · 4 AM 81°. rain drop 0% · 5 AM 81°. rain drop 0% · 6 AM 81°. rain ...', 'position': 1}, {'title': 'Weather Forecast and Conditions for Hyderabad, Telangana, India', 'link': 'https://weather.com/